FIAP

# TIAO - Machine Learning - Aula 05

## Preparação de dados + Métodos de ensemble + Interpretatibilidade


### Imports, leitura da base e análise exploratória

In [1]:
# Instalações
!pip install lime
!pip install shap

# Suprimir warnings
import warnings
warnings.filterwarnings('ignore')

# Leitura de dados e gráficos
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Preparação de dados
from imblearn.under_sampling import RandomUnderSampler
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

# Modelos preditivos
from sklearn.ensemble import BaggingClassifier, AdaBoostClassifier, RandomForestClassifier, VotingClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier

# Avaliação de modelos
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import confusion_matrix

# Interpretabilidade de modelos
from lime import lime_tabular
import shap



# Avaliando os modelos de classificação, calculando algumas métricas
def avaliar_modelo(y_true, y_pred):
    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred)
    recall = recall_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)
    return accuracy, precision, recall, f1

# Plota a matriz de confusão
def plotar_matriz_confusao(y_true, y_pred, title):

    # Matriz de confusão
    cm = confusion_matrix(y_true, y_pred)

    # Plot da matriz de confusão
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Não Saiu', 'Saiu'],
                yticklabels=['Não Saiu', 'Saiu'])
    plt.xlabel('Predição')
    plt.ylabel('Valor Real')
    plt.title(title)
    plt.show()

  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
   ---------------------------------------- 0.0/12.0 MB ? eta -:--:--
   ---- ----------------------------------- 1.3/12.0 MB 7.2 MB/s eta 0:00:02
   --------- ------------------------------ 2.9/12.0 MB 7.6 MB/s eta 0:00:02
   ---------------- ----------------------- 5.0/12.0 MB 8.6 MB/s eta 0:00:01
   ------------------------- -------------- 7.6/12.0 MB 9.5 MB/s eta 0:00:01
   ----------------------------------- ---- 10.7/12.0 MB 10.8 MB/s eta 0:00:01
   ---------------------------------------- 12.0/12.0 MB 11.0 MB/s  0:00:01
   ---------------------------------------- 0.0/2.1 MB ? eta -:--:--
   ---------------------------------------- 2.1/2.1 MB 16.5 MB/s  0:


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


   ---------------------------------------- 0.0/43.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/43.0 MB ? eta -:--:--
   ---------------------------------------- 0.3/43.0 MB ? eta -:--:--
   ---------------------------------------- 0.5/43.0 MB 995.5 kB/s eta 0:00:43
    --------------------------------------- 0.8/43.0 MB 1.2 MB/s eta 0:00:36
    --------------------------------------- 1.0/43.0 MB 1.3 MB/s eta 0:00:33
   - -------------------------------------- 1.3/43.0 MB 1.2 MB/s eta 0:00:34
   - -------------------------------------- 1.8/43.0 MB 1.4 MB/s eta 0:00:31
   - -------------------------------------- 2.1/43.0 MB 1.5 MB/s eta 0:00:29
   -- ------------------------------------- 2.6/43.0 MB 1.5 MB/s eta 0:00:27
   -- ------------------------------------- 2.9/43.0 MB 1.5 MB/s eta 0:00:27
   --- ------------------------------------ 3.4/43.0 MB 1.6 MB/s eta 0:00:25
   --- ------------------------------------ 3.7/43.0 MB 1.6 MB/s eta 0:00:25
   ---- ----------


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


ModuleNotFoundError: No module named 'imblearn'

In [ ]:
# CARREGAMENTO DOS DADOS
data = pd.read_csv('churn_modelling.csv')
display(data.sample(10))


# ANÁLISE EXPLORATÓRIA

#   - Informações gerais do dataset
display(data.info())

#   - Análises estatísticas gerais
display(data.describe())

#   - Identifcação de duplicatas
display(f"Número de linhas duplicadas: {data.duplicated().sum()}")

### Avaliando outliers via boxplot

In [ ]:
# Identificação de outliers via Boxplots:

#   - Selecionar as variáveis para o boxplot
features = ['Age', 'Tenure', 'Balance', 'NumOfProducts', 'EstimatedSalary']

#   - Criar uma figura com 2 linhas e 4 colunas
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

#   - Iterar pelas variáveis e criar o boxplot em cada subplot
for i, feature in enumerate(features):
    row = i // 3
    col = i % 3
    sns.boxplot(y=data[feature], ax=axes[row, col])
    axes[row, col].set_title(feature)

#   - Ajustar o layout da figura
plt.tight_layout()
plt.show()

### Correlação entre as features


In [ ]:
# Plot da matriz de correlação com mapa de calor
plt.figure(figsize=(12, 10))
sns.heatmap(data.corr(numeric_only=True),
            annot=True,
            cmap='coolwarm', fmt=".2f")
plt.title('Matriz de Correlação')
plt.show()

### Tratamento de dados, eliminando duplicatas e features irrelavantes para modelagem
- remoção de IDs
- não há features altamente relacionadas

In [ ]:
# Eliminando linhas duplicadas
data.drop_duplicates(inplace=True)

# Eliminando features que não fazem sentido para o problema
data.drop(columns=['RowNumber', 'CustomerId', 'Surname'], inplace=True)

### Avaliando o desbalanceamento dos dados

E tratando via UnderSampling da classe majoritária.

In [ ]:
# Plotar o desbalanceamento da variável "Exited"
plt.figure(figsize=(6, 4))
sns.countplot(x='Exited', data=data)
plt.title('Distribuição da variável Exited')
plt.xlabel('Exited')
plt.ylabel('Contagem')
plt.show()

In [ ]:
# Separar as features (X) e o target (y)
X = data.drop('Exited', axis=1)
y = data['Exited']

# Criar o objeto RandomUnderSampler
rus = RandomUnderSampler(sampling_strategy={0: int(y.value_counts()[0] * 0.4)}, random_state=42)

# Aplicar o undersampling aos dados
X_resampled, y_resampled = rus.fit_resample(X, y)

# Criar um novo DataFrame com os dados reamostrados
data_resampled = pd.concat([X_resampled, y_resampled], axis=1)

# Verificar a contagem de cada classe após o undersampling
plt.figure(figsize=(6, 4))
sns.countplot(x='Exited', data=data_resampled)
plt.title('Distribuição da variável Exited')
plt.xlabel('Exited')
plt.ylabel('Contagem')
plt.show()

### Feature engineering

- Aplicação de OneHotEncoder
- Separação de treino e teste
- Padronizar features

In [ ]:
# Convertendo variáveis categóricas para numéricas (se necessário)
X_resampled = pd.get_dummies(data_resampled, drop_first=True)


# Separando os dados em treino e teste
X_train, X_test, y_train, y_test = train_test_split(X_resampled.drop('Exited', axis=1),
                                                    X_resampled['Exited'],
                                                    test_size=0.2,
                                                    random_state=42)

# Padronizando as features
ss = StandardScaler()
X_train_scaled = ss.fit_transform(X_train)
X_test_scaled = ss.transform(X_test)

### Cria um modelo preditivo baseline baseado numa classificação linear.

Avalia os resultados em termos de métricas de matriz de confusão.

In [ ]:
# Modelo baseline (classificação linear)
baseline_model = LogisticRegression()
baseline_model.fit(X_train_scaled, y_train)
y_pred_baseline = baseline_model.predict(X_test_scaled)
accuracy_baseline, precision_baseline, recall_baseline, f1_baseline = avaliar_modelo(y_test, y_pred_baseline)
print(f"BASELINE: Acurácia: {accuracy_baseline:.4f}, Precisão: {precision_baseline:.4f}, Recall: {recall_baseline:.4f}, F1-Score: {f1_baseline:.4f}")

In [ ]:
plotar_matriz_confusao(y_test, y_pred_baseline, "Matriz de Confusão - Baseline")

### Avalia estratégias de ensemble baseado em classificadores menos sofisticados

Aplica as técnicas de:
- bagging
- boosting
- voting
- stacking

In [ ]:
# Bagging - Combinação de 100 árvores independentes (similar à Floresta)
bagging_model = BaggingClassifier(estimator=DecisionTreeClassifier(),
                                  n_estimators=100,
                                  random_state=42)
bagging_model.fit(X_train_scaled, y_train)
y_pred_bagging = bagging_model.predict(X_test_scaled)

# Boosting
boosting_model = AdaBoostClassifier(n_estimators=100,
                                    random_state=42)
boosting_model.fit(X_train_scaled, y_train)
y_pred_boosting = boosting_model.predict(X_test_scaled)

# Definindo modelos base para ensembles para VOTING e STACKING
base_models = [
      ('dt', DecisionTreeClassifier(random_state=42)),
      ('rf', RandomForestClassifier(n_estimators=100, random_state=42)),
      ('knn', KNeighborsClassifier()),
      ('svm', SVC())
]

# Voting - Votação entre os modelos de base
voting_model = VotingClassifier(estimators=base_models,
                                voting='hard')
voting_model.fit(X_train_scaled, y_train)
y_pred_voting = voting_model.predict(X_test_scaled)

# Stacking
stacking_model = StackingClassifier(estimators=base_models,
                                    final_estimator=LogisticRegression())
stacking_model.fit(X_train_scaled, y_train)
y_pred_stacking = stacking_model.predict(X_test_scaled)

# XGBoost
xgb_model = XGBClassifier(random_state=42)
xgb_model.fit(X_train_scaled, y_train)
y_pred_xgb = xgb_model.predict(X_test_scaled)


# Avaliação dos modelos
accuracy_bagging, precision_bagging, recall_bagging, f1_bagging = avaliar_modelo(y_test, y_pred_bagging)
accuracy_boosting, precision_boosting, recall_boosting, f1_boosting = avaliar_modelo(y_test, y_pred_boosting)
accuracy_voting, precision_voting, recall_voting, f1_voting = avaliar_modelo(y_test, y_pred_voting)
accuracy_stacking, precision_stacking, recall_stacking, f1_stacking = avaliar_modelo(y_test, y_pred_stacking)
accuracy_xgb, precision_xgb, recall_xgb, f1_xgb = avaliar_modelo(y_test, y_pred_xgb)

print(f"Bagging:\tAcurácia: {accuracy_bagging:.4f}, \
        Precisão: {precision_bagging:.4f}, \
        Recall: {recall_bagging:.4f}, \
        F1-Score: {f1_bagging:.4f}")
print(f"Boosting:\tAcurácia: {accuracy_boosting:.4f}, \
        Precisão: {precision_boosting:.4f}, \
        Recall: {recall_boosting:.4f}, \
        F1-Score: {f1_boosting:.4f}")
print(f"Voting:\t\tAcurácia: {accuracy_voting:.4f}, \
        Precisão: {precision_voting:.4f}, \
        Recall: {recall_voting:.4f}, \
        F1-Score: {f1_voting:.4f}")
print(f"Stacking:\tAcurácia: {accuracy_stacking:.4f}, \
        Precisão: {precision_stacking:.4f}, \
        Recall: {recall_stacking:.4f}, \
        F1-Score: {f1_stacking:.4f}")
print(f"XGBoost:\tAcurácia: {accuracy_xgb:.4f}, \
        Precisão: {precision_xgb:.4f}, \
        Recall: {recall_xgb:.4f}, \
        F1-Score: {f1_xgb:.4f}")

### Plotando as matrizes de confusão

In [ ]:
# Plota as matrizes de confusão para cada modelo
plotar_matriz_confusao(y_test, y_pred_baseline, "Matriz de Confusão - Baseline")
plotar_matriz_confusao(y_test, y_pred_bagging, "Matriz de Confusão - Bagging")
plotar_matriz_confusao(y_test, y_pred_boosting, "Matriz de Confusão - Boosting")
plotar_matriz_confusao(y_test, y_pred_voting, "Matriz de Confusão - Voting")
plotar_matriz_confusao(y_test, y_pred_stacking, "Matriz de Confusão - Stacking")
plotar_matriz_confusao(y_test, y_pred_xgb, "Matriz de Confusão - XGBoost")

### Interpretabilidade do melhor modelo

O melhor modelo foi o stacking.

In [ ]:
# LIME
explainer = lime_tabular.LimeTabularExplainer(
    X_train_scaled,
    feature_names=X_test.columns.tolist(),
    class_names=['Não Saiu', 'Saiu'],
    mode='classification'
)

# Escolha um índice da amostra de teste para explicar
i = 100
exp = explainer.explain_instance(
    X_test_scaled[i],
    stacking_model.predict_proba,
    num_features=10
)
exp.show_in_notebook(show_table=True)

In [ ]:
# De fato, não saiu!
y_test.iloc[100]

`stacking_model` é o melhor, mas demora muito! Vamos avaliar o XGBoost:

In [ ]:
# SHAP
explainer = shap.Explainer(xgb_model.predict, X_test_scaled)
shap_values = explainer(X_test_scaled)

# Resumo das importâncias das features
shap.summary_plot(shap_values, X_test_scaled, feature_names=X_test.columns.tolist())